### Cell 1 - 雨量測站資料前處理與空間化
---
- 本段程式碼主要用於載入兩場降雨事件（2024-07-25 與 2024-11-11）的雨量測站資料，並進行前處理與空間資料建構。首先，讀取原始 CSV 檔案後，將測站經緯度與過去 1 小時累積雨量（Past1hr）轉為數值格式，接著篩選出雨量大於 0 且排除異常值（-998）的測站資料。

- 為聚焦研究區域，程式進一步保留宜蘭縣與花蓮縣的測站，並檢查各事件的測站分布情形。之後，利用測站經緯度建立 GeoDataFrame，將原始座標系統由 WGS84（EPSG:4326）轉換為 TWD97 / TM2 zone 121（EPSG:3826），以利後續距離計算、空間分析與地統計插值。

- 最後，程式從空間資料中提取測站的 x、y 座標與雨量值 z，建立後續 variogram 分析與 Kriging 插值所需的 xyz 陣列。

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point
plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

import os
os.makedirs('output', exist_ok=True)


print("載入雨量資料...")

# 載入2024-07-25資料
df_0725 = pd.read_csv('data/rain_20240725.csv')
print(f"2024-07-25原始資料: {len(df_0725)} 筆")

# 載入2024-11-11資料  
df_1111 = pd.read_csv('data/rain_20241111.csv')
print(f"2024-11-11原始資料: {len(df_1111)} 筆")

# 轉換資料類型
df_0725['StationLatitude'] = pd.to_numeric(df_0725['StationLatitude'])
df_0725['StationLongitude'] = pd.to_numeric(df_0725['StationLongitude'])
df_0725['Past1hr'] = pd.to_numeric(df_0725['Past1hr'])

df_1111['StationLatitude'] = pd.to_numeric(df_1111['StationLatitude'])
df_1111['StationLongitude'] = pd.to_numeric(df_1111['StationLongitude'])
df_1111['Past1hr'] = pd.to_numeric(df_1111['Past1hr'])

# 過濾雨量>0且不等於-998
df_0725_filtered = df_0725[(df_0725['Past1hr'] > 0) & (df_0725['Past1hr'] != -998)]
df_1111_filtered = df_1111[(df_1111['Past1hr'] > 0) & (df_1111['Past1hr'] != -998)]

print(f"2024-07-25過濾雨量後: {len(df_0725_filtered)} 筆")
print(f"2024-11-11過濾雨量後: {len(df_1111_filtered)} 筆")

# 過濾宜蘭縣和花蓮縣
target_counties = ['宜蘭縣', '花蓮縣']
df_0725_counties = df_0725_filtered[df_0725_filtered['CountyName'].isin(target_counties)]
df_1111_counties = df_1111_filtered[df_1111_filtered['CountyName'].isin(target_counties)]

print(f"2024-07-25過濾縣市後: {len(df_0725_counties)} 筆")
print(f"2024-11-11過濾縣市後: {len(df_1111_counties)} 筆")

# 顯示縣市分布
print(f"\\n2024-07-25縣市分布: {df_0725_counties['CountyName'].value_counts().to_dict()}")
print(f"2024-11-11縣市分布: {df_1111_counties['CountyName'].value_counts().to_dict()}")

# 創建GeoDataFrame
geometry_0725 = [Point(lon, lat) for lon, lat in zip(df_0725_counties['StationLongitude'], df_0725_counties['StationLatitude'])]
gdf_0725 = gpd.GeoDataFrame(df_0725_counties, geometry=geometry_0725, crs='EPSG:4326')

geometry_1111 = [Point(lon, lat) for lon, lat in zip(df_1111_counties['StationLongitude'], df_1111_counties['StationLatitude'])]
gdf_1111 = gpd.GeoDataFrame(df_1111_counties, geometry=geometry_1111, crs='EPSG:4326')

# 轉換到EPSG:3826
gdf_0725_3826 = gdf_0725.to_crs('EPSG:3826')
gdf_1111_3826 = gdf_1111.to_crs('EPSG:3826')

print(f"\\n轉換EPSG:3826完成")
print(f"事件0725最終測站數: {len(gdf_0725_3826)}")
print(f"事件1111最終測站數: {len(gdf_1111_3826)}")

# 提取坐標和雨量作為xyz陣列
x_0725 = gdf_0725_3826.geometry.x.values  # 經度 (Easting)
y_0725 = gdf_0725_3826.geometry.y.values  # 緯度 (Northing)  
z_0725 = gdf_0725_3826['Past1hr'].values   # 雨量

x_1111 = gdf_1111_3826.geometry.x.values  # 經度 (Easting)
y_1111 = gdf_1111_3826.geometry.y.values  # 緯度 (Northing)
z_1111 = gdf_1111_3826['Past1hr'].values   # 雨量

print(f"\\n事件0725 xyz陣列:")
print(f"  x_0725 shape: {x_0725.shape}, 範圍: {x_0725.min():.0f} - {x_0725.max():.0f}")
print(f"  y_0725 shape: {y_0725.shape}, 範圍: {y_0725.min():.0f} - {y_0725.max():.0f}")
print(f"  z_0725 shape: {z_0725.shape}, 範圍: {z_0725.min():.1f} - {z_0725.max():.1f} mm/hr")

print(f"\\n事件1111 xyz陣列:")
print(f"  x_1111 shape: {x_1111.shape}, 範圍: {x_1111.min():.0f} - {x_1111.max():.0f}")
print(f"  y_1111 shape: {y_1111.shape}, 範圍: {y_1111.min():.0f} - {y_1111.max():.0f}")
print(f"  z_1111 shape: {z_1111.shape}, 範圍: {z_1111.min():.1f} - {z_1111.max():.1f} mm/hr")

### Cell 2 -半變異函數模型擬合與參數比較
---

- 本段程式碼針對兩場降雨事件的雨量測站資料，建立 Ordinary Kriging 所需的半變異函數（variogram）模型，並比較 spherical 與 exponential 兩種常見模型的擬合效果。首先，為降低雨量分布偏態的影響，程式先對 Past1hr 雨量資料進行 `log1p` 轉換，再分別建立兩種 variogram 模型。

- 接著，程式擷取實驗半變異值（experimental semivariance）與 lag distance，並計算各模型的參數，包括 Nugget、Sill 與 Range。同時，利用 SSE（sum of squared errors）評估理論曲線與實驗半變異值之間的擬合誤差，以判斷哪一種模型較適合描述各降雨事件的空間結構。

- 最後，程式將兩個事件的模型擬合結果繪製成圖，輸出各模型參數表，並進一步以 SSE 較小者作為每場事件的最佳模型，整理成跨事件比較表，以利後續分析不同降雨事件在空間變異特性上的差異。

In [ ]:
from pykrige.ok import OrdinaryKriging
from pykrige.variogram_models import spherical_variogram_model, exponential_variogram_model

events = [
    ('事件_1_20240725', x_0725, y_0725, z_0725),
    ('事件_2_20241111', x_1111, y_1111, z_1111),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
records = []

for row, (name, x, y, z) in enumerate(events):
    z_log = np.log1p(z)

    ok_sph = OrdinaryKriging(x, y, z_log, variogram_model='spherical',
                             nlags=20, verbose=False, enable_plotting=False)
    ok_exp = OrdinaryKriging(x, y, z_log, variogram_model='exponential',
                             nlags=20, verbose=False, enable_plotting=False)

    lags = ok_sph.lags
    sv   = ok_sph.semivariance
    d    = np.linspace(0, lags.max() * 1.05, 300)

    p_sph = ok_sph.variogram_model_parameters  # [psill, range, nugget]
    p_exp = ok_exp.variogram_model_parameters

    sse_sph = np.sum((sv - spherical_variogram_model(p_sph, lags)) ** 2)
    sse_exp = np.sum((sv - exponential_variogram_model(p_exp, lags)) ** 2)

    records.append({'事件': name, '模型': 'Spherical',
                    'Nugget': round(p_sph[2], 4), 'Sill': round(p_sph[0], 4),
                    'Range_m': round(p_sph[1], 1), 'Range_km': round(p_sph[1]/1000, 2),
                    'SSE': round(sse_sph, 6)})
    records.append({'事件': name, '模型': 'Exponential',
                    'Nugget': round(p_exp[2], 4), 'Sill': round(p_exp[0], 4),
                    'Range_m': round(p_exp[1], 1), 'Range_km': round(p_exp[1]/1000, 2),
                    'SSE': round(sse_exp, 6)})

    for col, (model_name, p, curve_func, color) in enumerate([
        ('Spherical',   p_sph, spherical_variogram_model,   'b'),
        ('Exponential', p_exp, exponential_variogram_model, 'r'),
    ]):
        sse = sse_sph if col == 0 else sse_exp
        ax = axes[row, col]
        ax.scatter(lags / 1000, sv, color='k', zorder=5, s=30, label='Experimental')
        ax.plot(d / 1000, curve_func(p, d), f'{color}-', linewidth=2, label=f'{model_name} Fit')
        ax.set_title(f'{name}\n{model_name}  |  SSE={sse:.5f}\n'
                     f'psill={p[0]:.3f}, range={p[1]/1000:.1f}km, nugget={p[2]:.3f}')
        ax.set_xlabel('Lag Distance (km)')
        ax.set_ylabel('Semivariance')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join('output/variogram_fitting_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()
print("圖表已儲存：variogram_fitting_comparison.png")

# 輸出 CSV
df_params = pd.DataFrame(records, columns=['事件', '模型', 'Nugget', 'Sill', 'Range_m', 'Range_km', 'SSE'])
df_params.to_csv(os.path.join('output/variogram_parameters.csv'), index=False, encoding='utf-8-sig')
print("\n參數比較表：")
print(df_params.to_string(index=False))
print("\nCSV已儲存：variogram_parameters.csv")

# A5 跨事件比較表
df_params = pd.DataFrame(records, columns=['事件', '模型', 'Nugget', 'Sill', 'Range_m', 'Range_km', 'SSE'])
df_params.to_csv(os.path.join('output/variogram_parameters.csv'), index=False, encoding='utf-8-sig')
print("\n參數比較表：")
print(df_params.to_string(index=False))
print("\nCSV已儲存：variogram_parameters.csv")

# ── A5 跨事件綜合比較 ────────────────────────────────────────────────────────
# 每個事件取 SSE 較小的模型作為 Best Model
rows_a5 = []
for event_name in [r['事件'] for r in records[::2]]:  # 每兩筆取一個事件名
    sph = next(r for r in records if r['事件'] == event_name and r['模型'] == 'Spherical')
    exp = next(r for r in records if r['事件'] == event_name and r['模型'] == 'Exponential')
    best = sph if sph['SSE'] <= exp['SSE'] else exp
    rows_a5.append({
        '參數':       'Sill',
        event_name:   best['Sill'],
    })

# 用樞紐方式整理成 參數 × 事件 的表格
event_names = [r['事件'] for r in records[::2]]
e1, e2 = event_names[0], event_names[1]

def best_for(event):
    sph = next(r for r in records if r['事件'] == event and r['模型'] == 'Spherical')
    exp = next(r for r in records if r['事件'] == event and r['模型'] == 'Exponential')
    return sph if sph['SSE'] <= exp['SSE'] else exp

b1, b2 = best_for(e1), best_for(e2)

df_a5 = pd.DataFrame([
    {'參數': 'Sill',        '事件1': b1['Sill'],       '事件2': b2['Sill'],       '差異原因': ''},
    {'參數': 'Range (km)',  '事件1': b1['Range_km'],   '事件2': b2['Range_km'],   '差異原因': ''},
    {'參數': 'Nugget',      '事件1': b1['Nugget'],     '事件2': b2['Nugget'],     '差異原因': ''},
    {'參數': 'SSE',         '事件1': b1['SSE'],        '事件2': b2['SSE'],        '差異原因': ''},
    {'參數': 'Best Model',  '事件1': b1['模型'],       '事件2': b2['模型'],       '差異原因': ''},
])

df_a5.to_csv('variogram_comparison.csv', index=False, encoding='utf-8-sig')
print("\nA5 跨事件比較：")
print(df_a5.to_string(index=False))
print("\nCSV已儲存：variogram_comparison.csv")






### Cell 3 - 插值分析函式與輸出工具準備
---

- 本段程式碼主要定義後續空間插值分析所需的函式與工具，包含網格建立、IDW 插值、Kriging 模型選擇，以及 GeoTIFF 輸出。首先，匯入 Ordinary Kriging、最近鄰插值、Random Forest、以及 rasterio 等相關套件，作為後續不同插值方法與結果輸出的基礎。

- 接著，程式定義 `make_grid()` 用於建立研究區域的規則格網；`idw_interpolate()` 用於執行反距離加權插值（IDW）；`best_kriging_model()` 則同時比較 spherical 與 exponential 兩種 variogram 模型，並以 SSE 較小者作為最佳 Kriging 模型。這樣可在正式插值前，先選擇較能反映空間變異特性的理論模型。

- 最後，`save_geotiff()` 函式用於將插值結果輸出為 GeoTIFF 格式，並自動調整陣列方向以符合 raster 座標系統要求。這些函式的建立，提供了後續不同插值方法比較、空間分布製圖，以及 GIS 匯出的基礎架構。

In [ ]:
from pykrige.ok import OrdinaryKriging
from pykrige.variogram_models import spherical_variogram_model, exponential_variogram_model
from scipy.interpolate import NearestNDInterpolator
from sklearn.ensemble import RandomForestRegressor
import rasterio
from rasterio.transform import from_origin
import time, warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

def make_grid(x, y, resolution=1000, buffer=5000):
    gx = np.arange(x.min() - buffer, x.max() + buffer, resolution)
    gy = np.arange(y.min() - buffer, y.max() + buffer, resolution)
    return gx, gy

def idw_interpolate(x, y, z, gxx, gyy, power=2, chunk=2000):
    xi, yi = gxx.ravel(), gyy.ravel()
    result = np.zeros(len(xi))
    for i in range(0, len(xi), chunk):
        d = np.sqrt((xi[i:i+chunk, None] - x[None, :])**2 +
                    (yi[i:i+chunk, None] - y[None, :])**2)
        d = np.where(d == 0, 1e-10, d)
        w = 1.0 / d**power
        result[i:i+chunk] = (w * z).sum(axis=1) / w.sum(axis=1)
    return result.reshape(gxx.shape)

def best_kriging_model(x, y, z_log, nlags=20):
    ok_sph = OrdinaryKriging(x, y, z_log, variogram_model='spherical',
                              nlags=nlags, verbose=False, enable_plotting=False)
    ok_exp = OrdinaryKriging(x, y, z_log, variogram_model='exponential',
                              nlags=nlags, verbose=False, enable_plotting=False)
    lags, sv = ok_sph.lags, ok_sph.semivariance
    sse_sph = np.sum((sv - spherical_variogram_model(ok_sph.variogram_model_parameters, lags))**2)
    sse_exp = np.sum((sv - exponential_variogram_model(ok_exp.variogram_model_parameters, lags))**2)
    return (ok_sph, 'Spherical', sse_sph) if sse_sph <= sse_exp else (ok_exp, 'Exponential', sse_exp)

def save_geotiff(array, gx, gy, filepath, crs='EPSG:3826'):
    """儲存 GeoTIFF，自動 flipud 修正 y 軸方向"""
    arr = np.flipud(array).astype(np.float32)
    transform = from_origin(gx.min(), gy.max(), 1000, 1000)
    with rasterio.open(filepath, 'w', driver='GTiff',
                       height=arr.shape[0], width=arr.shape[1],
                       count=1, dtype='float32',
                       crs=crs, transform=transform) as dst:
        dst.write(arr, 1)
    print(f"  已儲存: {filepath}")

print("函式定義完成")


### Cell 4 - 四種空間內插方法的建模與比較
---
- 本段程式碼針對兩場降雨事件，分別執行四種空間內插方法，包括最近鄰法（Nearest Neighbor）、反距離加權法（IDW）、普通克利金法（Ordinary Kriging），以及隨機森林回歸（Random Forest），以比較不同方法對雨量空間分布的模擬效果。

- 首先，程式利用前面定義的 `make_grid()` 建立規則格網，作為各種內插方法的預測範圍。接著，依序計算最近鄰、IDW、Kriging 與 Random Forest 的插值結果。其中，Kriging 會先根據對數轉換後的雨量資料，自動選擇較佳的 variogram 模型，再執行格網預測；Random Forest 則以測站座標作為自變數，建立非線性空間預測模型。

- 每場事件的各種插值結果都會儲存在 `results` 字典中，供後續分析與輸出使用。最後，程式將四種方法的預測結果繪製成 2×2 比較圖，並統一色階範圍，方便直接比較不同插值方法在降雨空間分布、平滑程度與局部變化表現上的差異。

In [ ]:
events = [
    ('事件_1_20240725', x_0725, y_0725, z_0725),
    ('事件_2_20241111', x_1111, y_1111, z_1111),
]

# 儲存結果供後續 cell 使用
results = {}

for event_name, x, y, z in events:
    print(f"\n{'='*60}\n{event_name}\n{'='*60}")
    gx, gy = make_grid(x, y)
    gxx, gyy = np.meshgrid(gx, gy)
    extent = [gx.min(), gx.max(), gy.min(), gy.max()]

    t0 = time.time()
    z_nn = NearestNDInterpolator(list(zip(x, y)), z)(gxx, gyy)
    print(f"  NN       完成 ({time.time()-t0:.1f}s)")

    t0 = time.time()
    z_idw = idw_interpolate(x, y, z, gxx, gyy, power=2)
    print(f"  IDW      完成 ({time.time()-t0:.1f}s)")

    t0 = time.time()
    ok, model_name, sse = best_kriging_model(x, y, np.log1p(z))
    z_krig_log, variance = ok.execute('grid', gx, gy)
    z_krig = np.expm1(z_krig_log)
    z_krig[z_krig < 0] = 0
    print(f"  Kriging  完成 ({time.time()-t0:.1f}s)  使用: {model_name}  SSE={sse:.5f}")

    t0 = time.time()
    rf = RandomForestRegressor(n_estimators=200, min_samples_leaf=3, random_state=42)
    rf.fit(np.column_stack([x, y]), z)
    z_rf = rf.predict(np.column_stack([gxx.ravel(), gyy.ravel()])).reshape(gxx.shape)
    print(f"  RF       完成 ({time.time()-t0:.1f}s)")

    results[event_name] = dict(x=x, y=y, z=z, gx=gx, gy=gy, extent=extent,
                                z_nn=z_nn, z_idw=z_idw, z_krig=z_krig,
                                z_rf=z_rf, variance=variance, model_name=model_name)

    vmax = np.percentile(np.concatenate([z_nn, z_idw, z_krig, z_rf], axis=None), 98)
    results[event_name]['vmax'] = vmax

    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    fig.suptitle(f'{event_name} — 四種內插方法比較', fontsize=15, fontweight='bold')

    for ax, title, grid in zip(axes.ravel(),
                                ['Nearest Neighbor', 'IDW (power=2)',
                                 f'Ordinary Kriging ({model_name})', 'Random Forest'],
                                [z_nn, z_idw, z_krig, z_rf]):
        im = ax.imshow(grid, extent=extent, origin='lower', cmap='YlOrRd', vmin=0, vmax=vmax)
        ax.scatter(x, y, c='k', s=8, zorder=5)
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Easting (m)')
        ax.set_ylabel('Northing (m)')
        plt.colorbar(im, ax=ax, shrink=0.8, label='mm/hr')

    plt.tight_layout()
    plt.savefig(f'output/interpolation_comparison_{event_name}.png', dpi=150, bbox_inches='tight')
    plt.show()


### Cell 5 -Kriging 與 Random Forest 插值結果差異比較
---
- 本段程式碼用於比較 Ordinary Kriging 與 Random Forest 兩種方法在降雨空間預測上的結果差異。對每一場降雨事件，程式先計算 Kriging 預測值與 Random Forest 預測值之差，並建立差值圖（Kriging − RF），以觀察兩種方法在不同空間位置上的預測偏差。

- 圖中左、中的子圖分別呈現 Kriging 與 Random Forest 的預測結果，右圖則顯示兩者差值的空間分布，其中正值代表 Kriging 預測較高，負值則代表 Random Forest 預測較高。為了讓差異圖更容易比較，色階範圍是依差值絕對值的 98 百分位數設定，以避免極端值過度影響視覺化效果。

- 最後，程式計算並輸出每場事件中兩種方法的平均絕對差異（mean absolute difference），作為整體預測差距的量化指標。此步驟有助於評估 Kriging 與 Random Forest 在空間分布型態與預測強度上的一致性與差異性。

In [ ]:
for event_name, r in results.items():
    diff = r['z_krig'] - r['z_rf']
    vmax_diff = np.percentile(np.abs(diff), 98)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle(f'{event_name} — Kriging vs Random Forest', fontsize=14)

    for ax, title, grid, cmap, vmin_, vmax_ in [
        (axes[0], f"Kriging ({r['model_name']})", r['z_krig'], 'YlOrRd', 0,          r['vmax']),
        (axes[1], 'Random Forest',                r['z_rf'],   'YlOrRd', 0,          r['vmax']),
        (axes[2], 'Kriging - RF',                 diff,        'RdBu_r', -vmax_diff, vmax_diff),
    ]:
        im = ax.imshow(grid, extent=r['extent'], origin='lower', cmap=cmap, vmin=vmin_, vmax=vmax_)
        ax.scatter(r['x'], r['y'], c='k', s=8, zorder=5)
        ax.set_title(title, fontsize=12)
        ax.set_xlabel('Easting (m)')
        ax.set_ylabel('Northing (m)')
        plt.colorbar(im, ax=ax, shrink=0.8, label='mm/hr')

    plt.tight_layout()
    plt.savefig(f'output/kriging_rf_difference_{event_name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"{event_name}  Kriging−RF 平均絕對差異: {np.mean(np.abs(diff)):.2f} mm/hr")


### Cell 6 -Kriging 預測結果與不確定性分析（Sigma Map）
---
- 本段程式碼針對每一場降雨事件，進一步呈現 Ordinary Kriging 的預測結果與其對應的不確定性分布。左圖顯示 Kriging 所估計的降雨空間分布，右圖則為 Kriging variance（亦可視為 Sigma Map），用來表示不同位置上的預測不確定性大小。

- 在空間統計中，Kriging variance 反映的是模型在各格網位置上的估計信心。一般而言，距離測站較近、周圍觀測資料較密集的區域，其 variance 較低，代表預測較可靠；相反地，遠離測站或位於測站分布稀疏區域的位置，variance 通常較高，表示模型對該區域的預測不確定性較大。

- 最後，程式會輸出每場事件的平均 variance，作為整體預測不確定性的量化指標。透過將降雨估計圖與 Sigma Map 並列比較，可以同時理解降雨空間分布特徵，以及模型在不同區域的預測可信度。

In [ ]:
for event_name, r in results.items():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f'{event_name} — Kriging 不確定性分析 (Sigma Map)', fontsize=14, fontweight='bold')

    im1 = axes[0].imshow(r['z_krig'], extent=r['extent'], origin='lower',
                          cmap='YlOrRd', vmin=0, vmax=r['vmax'])
    axes[0].scatter(r['x'], r['y'], c='k', s=8, zorder=5)
    axes[0].set_title(f"Kriging 降雨估計 ({r['model_name']})")
    axes[0].set_xlabel('Easting (m)')
    axes[0].set_ylabel('Northing (m)')
    plt.colorbar(im1, ax=axes[0], shrink=0.8, label='mm/hr')

    im2 = axes[1].imshow(r['variance'], extent=r['extent'], origin='lower', cmap='Blues', vmin=0)
    axes[1].scatter(r['x'], r['y'], c='r', s=8, zorder=5, label='測站')
    axes[1].set_title('Kriging Variance (Sigma Map)')
    axes[1].set_xlabel('Easting (m)')
    axes[1].set_ylabel('Northing (m)')
    axes[1].legend(fontsize=9)
    plt.colorbar(im2, ax=axes[1], shrink=0.8, label='Variance')

    plt.tight_layout()
    plt.savefig(f'output/sigma_map_{event_name}.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f"{event_name}  平均 Variance: {np.nanmean(r['variance']):.4f}")


### Cell 7 - 插值結果輸出為 GeoTIFF
---
- 本段程式碼將前面各場降雨事件的內插結果輸出為 GeoTIFF 格式，以便後續在 GIS 軟體中進行展示、疊圖分析與空間應用。針對每一場事件，程式會依據事件名稱擷取日期，並使用先前建立的格網座標資訊，分別輸出 Kriging 降雨估計圖、Kriging variance 圖，以及 Random Forest 降雨估計圖。

- 其中，`z_krig` 代表 Ordinary Kriging 的降雨預測結果，`variance` 代表 Kriging 的預測不確定性，而 `z_rf` 則為 Random Forest 的空間預測結果。這些結果皆透過 `save_geotiff()` 函式儲存為具有空間參考資訊的 raster 檔案，使其可直接匯入 QGIS、ArcGIS 等 GIS 平台進行後續分析。

- 透過 GeoTIFF 輸出，研究結果不僅能保留完整的空間分布資訊，也方便後續與行政區、河川、地形或其他環境圖層進行整合與比較。

In [ ]:
for event_name, r in results.items():
    date = event_name.split('_')[-1]  # 20240725 or 20241111
    gx, gy = r['gx'], r['gy']

    save_geotiff(r['z_krig'],    gx, gy, f'kriging_rainfall_{date}.tif')
    save_geotiff(r['variance'],  gx, gy, f'kriging_variance_{date}.tif')
    save_geotiff(r['z_rf'],      gx, gy, f'rf_rainfall_{date}.tif')

print("\n所有 GeoTIFF 儲存完成")
